In [6]:
import os
import pandas as pd
from sklearn.cluster import KMeans
import numpy as np
from enum import Enum
from ast import literal_eval
import math
import re
from GraphAnalyzer import GraphAnalyzer
import chart_studio.plotly as py
import plotly.graph_objs as go
import ipywidgets as widgets
import json
from plotly.subplots import make_subplots
from DiscreteGraph import GraphType
from Util import GetInt




In [7]:
from database import LoadAllUsers

	

maxUser =10
maxLevel =2

path = 'C:/Users/z5308/Desktop/VRTestingProject/data/small/'
df = LoadAllUsers(maxUser, path)
#df = LoadAllUsers(maxUser, None)
def GetAllUserData( level:int ):
	filter_df = df[df['levelDiff'] == level]
	return filter_df

def GetSelectedUserData( level:int, userStart:int, userTo:int):
	filtered_df = df[(df['userID'] >= userStart)  
					& (df['userID'] <= userTo ) 
					& (df['levelDiff'] == level  )]
	return filtered_df

def GetAllLevelFromUser(userID:int):
	filter_df = df[df['userID'] == userID]




Openning user0-turn1-lvl4.csv
Openning user0-turn3-lvl0.csv
Openning user0-turn4-lvl2.csv
Openning user0-turn7-lvl1.csv
Openning user0-turn12-lvl3.csv
Openning user0-turn15-lvl5.csv
Openning user1-turn3-lvl1.csv
Openning user1-turn5-lvl4.csv
Openning user1-turn6-lvl2.csv
Openning user1-turn7-lvl3.csv
Openning user1-turn8-lvl0.csv
Openning user1-turn9-lvl5.csv
Openning user1-turn16-lvl4.csv
Openning user1-turn17-lvl0.csv
Openning user2-turn0-lvl0.csv
Openning user2-turn1-lvl1.csv
Openning user2-turn3-lvl5.csv
Openning user2-turn11-lvl2.csv
Openning user2-turn14-lvl4.csv
Openning user2-turn15-lvl3.csv
Openning user3-turn3-lvl3.csv
Openning user3-turn4-lvl2.csv
Openning user3-turn8-lvl1.csv
Openning user3-turn9-lvl0.csv
Openning user3-turn10-lvl4.csv
Openning user3-turn13-lvl5.csv
Openning user4-turn4-lvl3.csv
Openning user4-turn6-lvl0.csv
Openning user4-turn8-lvl5.csv
Openning user4-turn9-lvl2.csv
Openning user4-turn12-lvl1.csv
Openning user4-turn15-lvl4.csv
Openning user5-turn4-lvl5.csv

In [ ]:
# get user range from text
from ast import List

s = "7, 3,6,1"
s = "sfgsd"

def GetUserRangeFromText(userRangeText):
    def AddNumToArr(fromNum, toNum):
        arr = []
        for i in range(fromNum, toNum+1):
            arr.append(i)
        return arr
    def ParseInt(elem):
        parsed_value = -1
        try:
                parsed_value = int(elem)
                #print(f"add {parsed_value}")
        except ValueError:
                print(f"Unable to parse '{elem}' as an integer.")
        return parsed_value
    range_arr= userRangeText.split(",")
    userRangeArr = []
    for r in range_arr:
        elems = r.split("-")
        
        if(len(elems) == 1):
            userRangeArr.append(ParseInt(elems[0]))
        elif(len(elems) ==2):
            userRangeArr += AddNumToArr(ParseInt(elems[0]),ParseInt(elems[1]))
        else:
            print(f"wrong format for {r}")
    #print(userRangeArr)
    return userRangeArr

def GetSelectedUserRangeData(level:int, userSelectedRange):
	filtered_df = df[(df['userID'].isin(userSelectedRange)) & (df['levelDiff'] == level)]
	return filtered_df

GetUserRangeFromText(s)  
# df2 = GetSelectedUserRangeData(level =1, userSelectedRange= arr)
# df2["userID"]

In [ ]:
# discrete graph

def CreateTrace( graphType:int, colname:str, df:pd.DataFrame, x_names):
	
	initial_visibile = True
	trace = None
	legendName = f"{colname}"
	#print(f"x_names {x_names}   df[colname] { df[colname]}")
	if(graphType == GraphType.barGraph.value):
	
		trace =  go.Bar( x = x_names,
						y = df[colname] ,
						name = legendName , 
						visible=initial_visibile,
						#width = 3,
				)
	elif (graphType == GraphType.lineGraph.value):

		trace =  go.Scatter( x = x_names,
							y = df[colname] ,
							name = legendName , 
							visible=initial_visibile,
				)
		
	elif(graphType == GraphType.scatterGraph.value):  # scatter
		trace =  go.Scatter( 
			x = x_names,
			y = df[colname] ,
			name = legendName , 
			mode='markers',
			#marker=dict(size=10, color=marker_colors, symbol= marker_styles),
			visible=initial_visibile,)
		print("create scatter")
	elif(graphType == GraphType.Histogram.value):
		trace = go.Histogram(df[colname], x = x_names, nbins = 5)
	else: #histogram or pie char
		trace = go.Pie(df[colname])
		

		
	return trace



In [ ]:
import plotly.express as px

def CreateSuccessRateGraph(df, userRange):
    def SuccessRate(selectedUser, df):
        filtered_df = df[(df['userID'] == selectedUser)]  
        tot = len(filtered_df['GameplayResult'])
        
        count =   ((filtered_df['GameplayResult'] > 0) ).sum()
        result = count /tot; 
        result = round(result, 2)
        #print(f"tot played {tot}  win{count} result{result}")
    
        return result

   
    allLevelSuccessRate=[SuccessRate(i, df) for i in userRange]
    #print(allLevelSuccessRate)
    x_names = [ f"user{x}"  for x in userRange]
   
    trace = go.Bar(x = x_names,  y = allLevelSuccessRate )
    return trace

# df2 = df

# bins = [1,50,120,200]
# def CreatePieChart(custom_bins, temp_df):

#     hist_values, bin_edges = np.histogram(temp_df['GameplayDur'], bins=custom_bins)
#     #x_names = [f"Rating ={custom_bins[i]}" for i in range(len(custom_bins)-1) ]
#     print(f"hist value{hist_values}, binedge{bin_edges}")

#     def SetBinName(bin_edges):
#         bin_names = []
#         if(len(bin_edges) <2):
#             return "Not enough bin edges"
        
#         for i in  range(1, len(bin_edges)):
#             bin_names += [f"{round(bin_edges[i-1],2)} <= x < { round(bin_edges[i],2)}"]

#         #bin_names += [f"x >= {round(bin_edges[-1],2)}"]
#         return bin_names

#     x_names = SetBinName(bin_edges) #bin egdge always has number of bin +1
#     hover_template = 'Category: %{label}<br>Value: %{value}<br>Percentage: %{percent}'

#     trace =  go.Pie( values=hist_values, labels=x_names, hovertemplate=hover_template)
#     return trace

# fig = go.Figure()
# # trace = CreatePieChart(bins, df2)

# userRange = [0,1,3,4]
# trace =CreateSuccessRateGraph(df, userRange)

# #trace = go.Box(y = df['DifficultyRating'],boxpoints='outliers', boxmean= True, name = "colname  ")

# # Create a histogram
# # colName = 'GameplayDur'
# # hover_template = colName+ ' Count: %{y}<br> Bin Range: %{x}<br>'
# # trace = go.Histogram( x = df2[colName], nbinsx = 5, hovertemplate=hover_template)
# fig.add_trace(trace)
# fig.show()
# Show the plot


In [11]:
from enum import Enum, IntEnum
class basicAlgorithm(IntEnum):
    Avg = 0
    Std =1
    Median =2
    NumPeak = 3
    SuccessRate = 4

def SuccessRate(selectedUser, df):
        filtered_df = df[(df['userID'] == selectedUser)]  
        tot = len(filtered_df['GameplayResult'])
        
        count =   ((filtered_df['GameplayResult'] > 0) ).sum()
        result = count /tot; 
        result = round(result, 2)
        #print(f"tot played {tot}  win{count} result{result}")
    
        return result
def Calculator(df, userID, algorithm, colName):
    filtered_df= df[df['userID'] == userID]
    
    result = 0
    if(algorithm == basicAlgorithm.Avg):
        result = filtered_df[colName].mean()
    elif(algorithm == basicAlgorithm.Median):
        result = filtered_df[colName].median()
    else:
        result = filtered_df[colName].std()
    return result
    
r = SuccessRate(1, df)

r2 = Calculator(df,1, basicAlgorithm.Avg, "DifficultyRating")

r2

2.4285714285714284

In [13]:
from enum import Enum

class Day(Enum):
    MONDAY = 1
    TUESDAY = 2
    WEDNESDAY = 3
    THURSDAY = 4
    FRIDAY = 5

input_string = "MONDAY"  # String representing the name of the enum type
try:
    enum_value = Day[input_string]  # Convert the string to the corresponding enum value
    print(f"Enum Value: {enum_value}")
except KeyError:
    print(f"'{input_string}' is not a valid enum value.")

Enum Value: Day.MONDAY


In [ ]:
graphSetting = None
with open("./graphSetting/SimpleGraph.json") as json_file:
        data = json.load(json_file)
        graphSetting = data["SimpleGraphSetting"]
        print(data['graphType'])

for i in df['userID'].unique():
    print(i)

In [ ]:
import numpy as np
import plotly.graph_objs as go

positions = np.array([
    [0, 0, 0],
    [3, 4, 2],
    [0, 1, 0],
    [10, 10, 4]
])

# Sample time data (in seconds)
time = np.array([0, 1, 2, 3])

# Calculate velocity as the rate of change of position
# Velocity is a 3D vector (vx, vy, vz)
delta_positions = np.diff(positions, axis=0)
delta_time = np.diff(time)
velocities = delta_positions / delta_time[:, np.newaxis]
print(velocities)

# fig = go.Figure()
# trace = go.Scatter3d(
#     x = [v[0] for v in velocities],
#     y = [v[1] for v in velocities],
#     z = [v[2] for v in velocities],
#     name = f" user"
# )
# fig.add_trace(trace)

# display(fig)


import numpy as np

# Sample 3D position data as an array of (x, y, z) points
position_data = np.array([
    [0, 0, 0],
    [3, 4, 2],
    [0, 1, 0],
    [10, 10, 4]
])

# Define a time array (you can use any parameter that represents the independent variable)
time = np.array([1,2,3,4])

# Calculate the derivative of the position data with respect to time
derivative = np.gradient(position_data, 3, axis=0)

# The 'derivative' variable now contains the derivatives of x, y, and z components
# at each corresponding time point in the 'time' array.

# print("Derivative Data:")
print(derivative)



In [ ]:
level = 0

graphSetting = None
with open("./graphSetting/SimpleGraph.json") as json_file:
		data = json.load(json_file)
		graphSetting = data["SimpleGraphSetting"]



def UpdateGraphByPercentile(level, top_percentage = 0.25, colName = 'GameplayDur', greaterThanPercentile = True):

	
	filter_df = None
	quantile = 1-top_percentage/100

	if(greaterThanPercentile):
			
			# Calculate the top th percentile
			Xth_percentile = df[colName].quantile(quantile)
			filter_df = df[(df[colName] >= Xth_percentile) & (df['levelDiff'] == level)]
	else:

			Xth_percentile = df[colName].quantile(quantile)
			filter_df = df[(df[colName] < Xth_percentile) & (df['levelDiff'] == level)]
	
	return filter_df

def UpdateGraphByRank(Rank = 1, colName = 'DifficultyRating'):
	ascending = False
	selected_df = df[df['levelDiff'] == level]
	selected_df['Rank'] = selected_df[colName].rank(ascending=ascending, method='min')
	distinct_ranks = selected_df['Rank'].unique()
	print(distinct_ranks)
	filter_df = selected_df[(selected_df['Rank'] == Rank)]

	return filter_df

def DiscreteGraph(filter_df):
	traces = []
	fig = go.Figure()
	
	x_names = [ f"user{x}_turn{y}"  for x, y in zip(filter_df['userID'], filter_df['turn'] )]
	
	for col  in graphSetting:
		colName = col["Label"]
		trace = CreateTrace(col["graphType"], colName, filter_df, x_names )
		traces+=[trace]
		fig.add_trace(trace)
	print(f"traces len {len(traces)}") # num label * num level
	# Create and add slider
	
	fig.update_layout(
		#sliders = levelSelectionSliders,
		title = f'Selected Level\'s User Performance ',
		xaxis =dict(title='Users'),
		yaxis =dict(title='Rating Scores'),
		
	)

	return fig

# find users whose score is lower than 80% of the group ::  Percentile 0.8, greatherThan False
# find users whose score is higher than 80% of the group :: Percentile 0.2, greatherThan True
filter_df = UpdateGraphByPercentile(level, 45, greaterThanPercentile = False)
#filter_df = UpdateGraphByRank(Rank = 3)
fig = DiscreteGraph(filter_df)

display(fig)




In [ ]:
# Define the string that encodes the calculation
calculation_string = "S = (X * 1 - Y * 0.5)/2"

# Define the values of X and Y
X = np.array([10, 20, 30])
Y = np.array([5, 10, 15])

# Create a dictionary to hold the variable values
variables = {'X': X, 'Y': Y}

# Use eval to execute the calculation in the context of the variables dictionary
try:
    exec(calculation_string, variables)
    result = variables['S']
    print(f"Result: {result}")
except Exception as e:
    print(f"Error: {e}")

In [ ]:
# continuous graph
import plotly.express as px


def GetSelectedUserData( level:int, userStart:int, userTo:int):
    filtered_df = df[(df['userID'] >= userStart)  
                    & (df['userID'] <= userTo ) 
                    & (df['levelDiff'] == level  )]
    return filtered_df


def DetermineGraphType(df, userID, turn, showIndexArr):

    trace = None
    if(len(showIndexArr) == 3):
        trace = go.Scatter3d(
            x = [ pos[0] for pos in df['plyr_pos']],
            y = [ pos[1] for pos in df['plyr_pos']],
            z = [ pos[2] for pos in df['plyr_pos']],
            name = f'user_{str(userID)}_t{turn}',
        )
        print("add 3d scatter plot")
        return trace
    elif(len(showIndexArr) == 2):
        firstIndex = showIndexArr[0]
        secondIndex = showIndexArr[1]
        trace = go.Scatter( 
                x = [ pos[firstIndex] for pos in df['plyr_pos']],
                y = [ pos[secondIndex] for pos in df['plyr_pos']],
                
                legendgroup=f'group_{str(userID)}',
                name = f'user_{str(userID)}_t{turn}',
                marker_colorscale = 'Picnic',
                mode='markers',
            )
    
    else: # len <=1 or >3
        print("Error")

    return trace
        
def PositionGraph(df:pd.DataFrame, fig:go.Figure, rowIdx:int):
    
    for index, row in df.iterrows():

            userID = row['userID']
            turn = row['turn']
            trace = DetermineGraphType(row, userID, turn, [0,2])
            if(trace != None):
                #fig.add_trace(trace)
                fig.add_trace(trace, row = rowIdx, col = 1)
            #print(f"{userID}, {turn}")
    return fig, rowIdx+1

def CalHeadRot(headArr, skip:int):
        head = headArr[::skip]
        angleDiff = []
        for frame in range(1, len(head)):
            #angle = np.abs(head[frame][1] - head[frame -1][1]) 
            angle = head[frame][1] - head[frame -1][1]
            angleDiff += [angle]
        
        return angleDiff


def RotationGraph(df:pd.DataFrame, fig:go.Figure, rowIdx:int, skipFrame):

    for index, row in df.iterrows():
            userID = row['userID']
            turn = row['turn']
            headRotArr = CalHeadRot(row['plyr_rot'], skipFrame)
            trace = go.Scatter( 
                                x = [ i for i in range(len(headRotArr)) ],
                                y =  headRotArr,
                                
                                name =  f'user_{str(userID)}_t{turn}',
                                legendgroup=f'group_{str(userID)}',
                                #showlegend= False,
                                # mode='lines',
                                # line=dict(
                                #     color=px.colors.sequential.Viridis[index],
                                #     width=1
                                # ),
                                marker=dict(size=15, colorscale = 'Picnic'),
                                
                                )
            #print(f"head row {headRotArr[:5]}")
            fig.add_trace(trace, row = rowIdx, col = 1)
            #fig.add_trace(trace)
    return fig, rowIdx+1
    
def CountNumGraph(graphSetting):
    count = 0
    for graph in graphSetting['GraphSetting']:
        if(graph['Enabled']): 
            count = count +1
    return count

def GetSubplotType(graphSetting):
    typeArr = []
    subplotTittles = []
    for graph in graphSetting['GraphSetting']:
        if(graph['Enabled']): 
            
            subplotTittles.append( f"{graph['graphProperty']['GraphTitle'] }" )
            
            if(graph['graphProperty']['is3DGraph']):
                typeArr.append ([{'type': 'scatter3d'}]) 
            else: 
                typeArr.append ([{'type': 'xy'}]) 
    #print(subplotTittles)
    return typeArr, subplotTittles

def UpdateSubplotTitles(fig:go.Figure, graphSetting):
    colIdx = 1
    rowIdx = 1
    for graph in graphSetting['GraphSetting']:
        xTitle = graph['graphProperty']['xTitle']
        yTitle = graph['graphProperty']['yTitle']
        if(graph['Enabled']):
            fig.update_xaxes(title_text=xTitle, row=rowIdx, col=colIdx)
            fig.update_yaxes(title_text=yTitle, row=rowIdx, col=colIdx)
            rowIdx = rowIdx +1


class AttributeType(Enum):
        WalkingPath =0
        PlayerRot =1
        WalkDistance = 2
        Velocity =3
        Acceleration =4


In [ ]:

def DistVelAccGraph(df:pd.DataFrame, fig:go.Figure,rowIdx:int, DVAGraph_Toogle):
    
    #fig2= go.Figure()
    for index, row in df.iterrows():
            curRowIdx = rowIdx
            userID = row['userID']
            turn = row['turn']
            distance, velocities, acceleration = CalDistVelAcc(row['plyr_pos'], 20)
            trace1 = go.Scatter(
                x = [ i for i in range(len(distance)) ],
                y =  distance,
                name =  f'user_{str(userID)}_t{turn}_dist',
                legendgroup=f'group_{str(userID)}',
                marker=dict(size=5, colorscale = 'Picnic'),
            )
            trace2 = go.Scatter(
                x = [ i for i in range(len(velocities)) ],
                y =  velocities,
                name =  f'user_{str(userID)}_t{turn}_vel',
                legendgroup=f'group_{str(userID)}',
                marker=dict(size=5, colorscale = 'Picnic'),
            )
            trace3 = go.Scatter(
                x = [ i for i in range(len(acceleration)) ],
                y =  acceleration,
                name =  f'user_{str(userID)}_t{turn}_acc',
                legendgroup=f'group_{str(userID)}',
                marker=dict(size=5, colorscale = 'Picnic'),
            )

            if(DVAGraph_Toogle[0]):
                fig.add_trace(trace1, row = curRowIdx, col = 1)
                curRowIdx = curRowIdx+1
                
            if(DVAGraph_Toogle[1]):
                fig.add_trace(trace2, row = curRowIdx, col = 1)
                curRowIdx = curRowIdx+1
            if(DVAGraph_Toogle[2]):
                fig.add_trace(trace3, row = curRowIdx, col = 1)
                curRowIdx = curRowIdx+1

    return fig, rowIdx+1

def CalDistVelAcc(posArr, skip:int):
    pos = posArr[::skip]
    # time in second 
    # time = np.arange(0, len(posArr), skip)
    # time = time / skip
    #dt = np.diff(time)
    distance_vect = np.diff(np.array(pos), axis = 0) 

    distance_mag = np.linalg.norm(distance_vect, axis =1)

    acc_dist = 0
    accumulate_distArr = []
    for v_m in distance_mag:
        acc_dist = acc_dist + v_m
        accumulate_distArr +=[acc_dist]
 
    
     # delta time bet each point = 0.05(fps) * #skip frames
    dt = np.full((len(pos) -1), skip*0.05 )
    
# 
    
    
    #print("time diff", dt)
    #dt[:, np.newaxis] for  each elem in dt, create new row
    # each v in distance vec =>  distance vec[i] / dt[i]
    velocities = distance_vect / dt[:, np.newaxis]
    #velocities2 = np.gradient(np.array(pos), axis=0)
    velocities_n = np.linalg.norm(velocities, axis =1)

   
    
    dt_velocity = dt[:-1]
    accelerations = np.diff(velocities, axis=0) / dt_velocity[:, np.newaxis]
    #print(accelerations)
    # acceleration = np.gradient(velocities, axis=0)
    accelration_n = np.linalg.norm(accelerations, axis =1)
   
    # print(accumulate_distArr)
    # print(accelration_n)
    return accumulate_distArr, velocities_n, accelration_n

def GetToogleForDistVelAcc(graphSetting):
    boolArr= [False,False, False]
    for graph in graphSetting['GraphSetting']:
        if(graph['Enabled'] and graph['Attribute'] == AttributeType.WalkDistance.value):
                boolArr[0] = True
        if(graph['Enabled'] and graph['Attribute'] == AttributeType.Velocity.value):
                boolArr[1] = True
        if(graph['Enabled'] and graph['Attribute'] == AttributeType.Acceleration.value):
                boolArr[2] = True
    return boolArr

         
            

def ContinousGraph(graphSetting):      
    graph = graphSetting['GraphSetting'][0]
    selectedLevel:int = graphSetting['selectedLevel']
    SelectedUserRange_Start:int = graphSetting["SelectedUserRangeStart"]
    SelectedUserRange_End:int = graphSetting['SelectedUserRangeEnd']

    selectedUserRange =[SelectedUserRange_Start, SelectedUserRange_End ]

    selectedUserRange = [0,1]
    showIndex = [0,2]
    skipFrame = 20
    nextRow:int =1
    specsArr, subplotTitles = GetSubplotType(graphSetting)
    numGraph = CountNumGraph(graphSetting)
    print(f"numGraph {numGraph}")
    if(numGraph <1):
        print("no attribute has not been enabled")
    fig:go.Figure = make_subplots(rows = numGraph, cols =1,
                subplot_titles = subplotTitles, specs= specsArr)

    filter_df:pd.DataFrame = GetSelectedUserData(selectedLevel, selectedUserRange[0], selectedUserRange[1])

    
    UpdateSubplotTitles(fig, graphSetting)
    


    for graph in graphSetting['GraphSetting']:
        
        if(graph['Enabled'] and graph['Attribute'] == AttributeType.WalkingPath.value):
            
            fig, nextRow = PositionGraph(filter_df,fig, nextRow) 
        
        elif(graph['Enabled'] and graph['Attribute'] == AttributeType.PlayerRot.value):
            fig, nextRow = RotationGraph(filter_df,fig, nextRow, skipFrame)
        

    DVAGraph_Toogle = GetToogleForDistVelAcc(graphSetting)
       
    fig, nextRow = DistVelAccGraph(filter_df, fig, nextRow, DVAGraph_Toogle)
        # else:
        #     print(f"[Error] no graph attribute for this int  {graph['Attribute']}") 

    figureWidth =  GetInt(graphSetting['figureWidth'], 800) 
    figureHeight = GetInt(graphSetting['figureHeight'], 280)* numGraph
    fig.update_layout(
            width=figureWidth, height=figureHeight,
            title = f'Selected Level\'s User Performance',
        )

    return fig


#get setting from json
graphSetting = None
with open("./graphSetting/ContinuousGraph.json") as json_file:
	graphSetting = json.load(json_file)

fig = ContinousGraph(graphSetting)
display(fig)


In [ ]:
import plotly.express as px
class AttributeType(Enum):
    PlayerPos =0
    PlayerRot =1


In [ ]:
#clustering

from enum import IntEnum
from plotly.subplots import make_subplots
from scipy.signal import find_peaks
class basicAlgorithm(IntEnum):
    Avg = 0
    Std =1
    Median =2
    NumPeak =3

from typing import List, Tuple

maxLevel = 4

def FindPeak(arr):
    
    peaks, properties  = find_peaks(arr, distance=30, height=1)
    peak_yloc =[arr[p] for p in peaks]

    return peaks, len(peaks)
def CalHeadRot(headArr, skip:int):
    head = headArr[::skip]
    angleDiff = []
    for frame in range(1, len(head)):
        angle = np.abs(head[frame][1] - head[frame -1][1]) 
        angleDiff += [angle]
    
    return angleDiff
def Calculator(df, colName, selectedLevel:int, algorithm: basicAlgorithm):
    result = 0
    if(colName == "sucessRate"):
        tot = (df['levelDiff'] == selectedLevel).sum()
        count =   ((df['GameplayResult'] > 0) &  (df['levelDiff'] == selectedLevel)).sum()
        result = count /tot; 
        result = round(result, 2)
       
    elif(colName == "playerRot"):
        filtered_df= df[df['levelDiff'] == selectedLevel]
        results = []
        for index, row in filtered_df.iterrows():
            userID = row['userID']
            turn = row['turn']
            headRotArr = CalHeadRot(row['plyr_rot'], 20)
            headRot_df = pd.DataFrame(headRotArr)

            if(algorithm == basicAlgorithm.Avg):
                results += [headRot_df.mean()[0]]
            elif(algorithm == basicAlgorithm.Median):
                results += [headRot_df.median()[0]]
            elif(algorithm == basicAlgorithm.Std):
                results += [headRot_df.std()[0]]
            else:
                _, numPeak = FindPeak(headRotArr)
                results += [numPeak]
            #print(f"{userID} ::{headRot_df.mean()[0]}")
        result = pd.DataFrame(results).mean()[0] 
        
        # print(f"  palyerRot::{result}")
        
    else:
        
        
        filtered_df= df[df['levelDiff'] == selectedLevel]
        if(algorithm == basicAlgorithm.Avg):
            result = filtered_df[colName].mean()
        elif(algorithm == basicAlgorithm.Median):
            result = filtered_df[colName].median()
        else:
            result = filtered_df[colName].std()
    
    result = round(result, 2)
    return result


def constructDF_Cluser(df, feature_alg_pairs: List[ Tuple[str, basicAlgorithm]]):
    #colArr = [pair[0]  for pair in feature_alg_pairs] + ["level"]
    colNameArr =[ f"{pair[1].name}_{pair[0]}"  for pair in feature_alg_pairs]
    # level_df = pd.DataFrame(columns= colArr)
    level_df = pd.DataFrame({})
    count = 0

    resultArr = []
   
    for level in range(maxLevel):
        #calculated success rate
        new_data = {}
        for pair in feature_alg_pairs:
            featureName = pair[0]
            method = pair[1]
           
            key = f"{method.name}_{featureName}"
            new_data[key] = Calculator(df, featureName, level, method)
        
        new_df = pd.DataFrame([new_data])

        level_df = pd.concat([level_df, new_df], ignore_index = True)
        
   
    return level_df, colNameArr

def Run_Cluser(num_clusters, level_df, colNameArr):
    model = KMeans(n_init = 'auto', n_clusters=num_clusters)
    #arr = level_df[['successRate', 'avgDur', 'avg_diffRating']]
    arr = level_df[[ item for item in colNameArr]]
    
    model.fit(arr)
    predict = model.predict(arr) 
    level_df['predictGroup'] = predict
   
    return level_df,'predictGroup'
  

def ClusterGraph(df,colNameArr, predictedColName, graphSetting):
    hoverTextName = [ f"level{x}"  for x in range( df.shape[0] )]
    trace = None
    fig =  make_subplots(
        rows=2, cols=1,
        shared_xaxes=False,
        vertical_spacing=0.1,
        specs=[
            [{"type": "scatter3d"}] if len(colNameArr) >2 else [{"type": "scatter"}],
            [{"type": "table"}]]
        )
    
    if(colNameArr is None or len(colNameArr) <1):
        print("[Error] colName Arr is empty")
        return
    
    
    mainTitle = graphSetting['mainTitle']
    xTitle = graphSetting['xTitle']
    yTitle = graphSetting['yTitle']
    zTitle = 'undefined' if 'zTitle' not in graphSetting else graphSetting['zTitle']
    
    if(len(colNameArr) == 1):
        trace = go.Scatter(   
                    x = df.index,
                    y = df[colNameArr[0]],
                    mode='markers+text',
                    marker=dict(size=10, color= df[predictedColName], colorscale = 'Picnic'),
                    visible=True,
                    text = hoverTextName,
                    hovertext = hoverTextName )
    elif(len(colNameArr) == 2):
        trace =  go.Scatter( 
                    x = df[colNameArr[0]] ,
                    y = df[colNameArr[1]],
                    text = hoverTextName,
                    mode='markers+text',
                    marker=dict(size=10, color= df[predictedColName], colorscale = 'Picnic'),
                    visible=True,
                    hovertemplate='xTitle'+ '=%{x}<br>' + yTitle 
                    + '=%{y}<br>'+zTitle+ ' =%{z}',
                    hovertext = hoverTextName ) 
    else:
        trace =  go.Scatter3d( 
                    x = df[colNameArr[0]] ,
                    y = df[colNameArr[1]],
                    z = df[colNameArr[2]],
                    text = hoverTextName,
                    mode='markers+text',
                    marker=dict(size=10, color= df[predictedColName], colorscale = 'Picnic'),
                    visible=True,
                    hovertemplate=xTitle+ '=%{x}<br>' + yTitle 
                                + '=%{y}<br>'+zTitle+ ' =%{z}<br>'+ '%{hovertext}',
                    hovertext = hoverTextName )
    
    fig.add_trace(trace, row=1, col=1)
    if(len(colNameArr) <3):
        fig.update_layout(
                title= mainTitle,
                xaxis=dict(title=xTitle),
                yaxis=dict(title=yTitle),
                width = 600,
                )
    else:
         fig.update_layout(
                title= mainTitle,
                scene=dict(
                xaxis=dict(title=xTitle),
                yaxis=dict(title=yTitle),
                zaxis= dict(title = zTitle)),
                width = 900,
                height = 800,
                )
    df['LevelIdx'] = df.index
  
    table = go.Table(
        header=dict(values= df.columns,
                    fill = dict(color='#C2D4FF'),
                    align = ['center'] * 5),
        cells=dict(values= [df[col] for col in df.columns],
                fill = dict(color='#F5F8FF'),
                align = ['center'] * 5),
        
                )

   

    fig.add_trace(table, row=2, col=1)
   
    def Construct_buttons(df):
        buttons = []
        for selCol in  df.columns:
            button = dict(
                label= f'sort by {selCol}',
                method="restyle",
                args=[dict(
                    cells = { "values":  [df.sort_values(by = selCol)[col] for col in  df.columns]}
                )],
       
            )
            buttons.append(button)
            
        return buttons
   
   
    tableLayout = go.Layout(
        #title = "User Performance",
        updatemenus=[
        {
            'buttons' : Construct_buttons(df),
            'direction': 'down',
            'showactive': True,
            'x': 0,
            'xanchor': 'left',
            'y':0.5,
            'yanchor': 'top'
        },
       
    ]
    )
    fig.update_layout(tableLayout) 

    return fig

 

In [ ]:
graphSetting = None
with open("./graphSetting/ClusterGraph.json") as json_file:
	graphSetting = json.load(json_file)

for item in graphSetting["GraphSetting"]:
	#print(item)
	...

N_cluster = graphSetting['clusterAlgSetting']['N_cluster']
feature_alg_pairs = [ (item['name'], basicAlgorithm(item['method']))  for item in graphSetting["GraphSetting"]]
level_df, colNameArr = constructDF_Cluser(df,feature_alg_pairs )

# temp = [colNameArr[0], colNameArr[1]]


level_df, clusterResultName = Run_Cluser(num_clusters=N_cluster, level_df = level_df, colNameArr = colNameArr)
fig = ClusterGraph(level_df, colNameArr, clusterResultName, graphSetting)
display(fig)


In [40]:
from Util import GetFloat

def filterDf(df, target_values, comparator_values, textfield_values):
        def RunComparator(compare_text, df, target, value):
            filter_df = None
            if(compare_text == ">"):
                filter_df = df[df[target] > value ]
            elif(compare_text == ">="):
                filter_df = df[df[target] >= value ]
            elif(compare_text == "=="):
                filter_df = df[df[target] == value ]
            elif(compare_text == "<"):
                filter_df = df[df[target] < value ]
            else: #if(compare_text == "<="):
                filter_df = df[df[target] <= value ]
            return filter_df

        filter_df = df
        for i in range(len(target_values)):
            target = target_values[i]
            compare_text = comparator_values[i]
            value =  GetFloat(textfield_values[i], 0) 
            if(target == None or compare_text == None):
                continue
            filter_df = RunComparator(compare_text, filter_df, target, value)
        
            
        return filter_df

target_values = ["GameplayDur", None, "DifficultyRating"]
comparator_values = [">", "<", "<"]
textfield_values = ["20.0", "0.0", "2.0"]
#filterDf(df, target_values, comparator_values, textfield_values)


def UpdateGraphByRank(df, comparator, rank = 1, colName = 'DifficultyRating'):
        ascending = True
        if(comparator == "Top"):
            ascending = False
        df['Rank'] = df[colName].rank(ascending=ascending, method='min')
        filter_df = df[(df['Rank'] < rank)]
   
        return  filter_df
filter_df = UpdateGraphByRank(df, "Top", 4)

filter_df[["userID","levelDiff","GameplayDur", "Rank"]]


,userID,levelDiff,GameplayDur,Rank
5,0,5,96.0,1.0
14,2,0,21.0,3.0
15,2,1,58.0,3.0
16,2,5,23.0,1.0
17,2,2,39.0,3.0
18,2,4,31.0,3.0
22,3,1,42.0,3.0
32,5,5,7.0,3.0


In [29]:
def GetNumber(s, default_v):
    try:
        v = int(s)
        # print(f"The value is an integer: {s}")
        return v
    except ValueError:
         try:
            return float(s)
         except ValueError:
            print(f"Invalid data type or input {s}, return default value {default_v}")
            return default_v
        # match = re.search(r'\d+\.\d+', s)
        # number = default_v
        # if match:
        #     number = int(match.group())
        # else:
        #   print(f"Invalid data type or input {s}, return default value {default_v}")
       
    #return number


GetNumber("16s", -1)

Invalid data type or input 16s, return default value -1


-1

In [12]:
import pandas as pd
df1 = pd.read_csv('C:/Users/z5308/Desktop/VRTestingProject/data/Reflex_data/simple_3joints/discreteInfo.csv')
df2 = pd.read_csv('C:/Users/z5308/Desktop/VRTestingProject/data/Reflex_data/simple_3joints/groupInfo.csv')

    
final_df = pd.merge(df1, df2, on='userID', how='inner', validate= 'one_to_one')

print(final_df)
# Save the updated CSV file
final_df.to_csv('C:/Users/z5308/Desktop/VRTestingProject/data/Reflex_data/simple_3joints/FinalInfo.csv', index=False)


   userID  GameplayDur  ratingA  ratingB    GroupID
0       0           30        3        5          1
1       1           20        2        2          1
2       3           27        1        5          0
